# Uniswap V3 LP 相場分析ツール

**分析対象:**
- ETH/USDC (Base)
- BTC/ETH (Base)
- BNB/USDT (BSC)
- XRP/USDT (BSC)

**実行順序:** Cell 1 → Cell 2（LPレンジ編集）→ Cell 3（FRED APIキー設定）→ Cell 4〜7（関数定義）→ Cell 8（マクロデータ取得）→ Cell 9（全チャート表示）

In [ ]:
# Cell 1: インポート
# 初回のみ: pip install -r requirements.txt を実行してください

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

import ccxt
import yfinance as yf
import pandas as pd
import pandas_ta as ta
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from fredapi import Fred
from datetime import datetime, timedelta, timezone

print(f"ccxt        : {ccxt.__version__}")
print(f"pandas      : {pd.__version__}")
print(f"pandas_ta   : {ta.version}")
print(f"yfinance    : {yf.__version__}")
print(f"numpy       : {np.__version__}")
print("\n✅ インポート完了")

In [ ]:
# Cell 2: テクニカル設定（ユーザー編集セル①）
# ここを編集して現在のLPポジション範囲を入力してください

PAIRS = {
    "ETH/USDC": {
        "binance_symbols": ["ETH/USDT"],
        "lp_lower": 2800.0,
        "lp_upper": 3600.0,
        "label": "ETH/USDC (Base)",
        "price_decimals": 2,
    },
    "BTC/ETH": {
        # BTC/USDT ÷ ETH/USDT のレシオで自動計算
        "binance_symbols": ["BTC/USDT", "ETH/USDT"],
        "lp_lower": 18.5,
        "lp_upper": 24.0,
        "label": "BTC/ETH (Base)",
        "price_decimals": 4,
    },
    "BNB/USDT": {
        "binance_symbols": ["BNB/USDT"],
        "lp_lower": 520.0,
        "lp_upper": 680.0,
        "label": "BNB/USDT (BSC)",
        "price_decimals": 2,
    },
    "XRP/USDT": {
        "binance_symbols": ["XRP/USDT"],
        "lp_lower": 0.48,
        "lp_upper": 0.72,
        "label": "XRP/USDT (BSC)",
        "price_decimals": 5,
    },
}

TIMEFRAME    = "4h"   # "1h" / "4h" / "1d"
CANDLE_LIMIT = 200    # 取得本数（Binance最大500）

BB_PERIOD  = 20
BB_STD     = 2.0
ADX_PERIOD = 14
ATR_PERIOD = 14

print("✅ テクニカル設定完了")

In [ ]:
# Cell 3: マクロ設定（ユーザー編集セル②）
# FRED APIキーを取得して設定: https://fred.stlouisfed.org/docs/api/api_key.html

FRED_API_KEY = "your_fred_api_key_here"  # ← ここに入力

# FOMC開催予定日（毎年1月頃に更新）
FOMC_DATES = [
    "2026-01-29", "2026-03-19", "2026-05-07",
    "2026-06-17", "2026-07-29", "2026-09-16",
    "2026-10-28", "2026-12-09",
]

# CME FedWatch（公式APIなしのため手動入力）
# 参照: https://www.cmegroup.com/markets/interest-rates/cme-fedwatch-tool.html
CME_RATE_HIKE_PROB_YE = 60.0  # 年末利上げ織り込み %
CME_RATE_HIKE_PROB_3M = 30.0  # 直近3ヶ月以内利上げ %

# yfinance シンボル（変更不要）
MACRO_YFINANCE = {
    "DXY":   "^DXY",
    "VIX":   "^VIX",
    "WTI":   "CL=F",
    "Gold":  "GC=F",
    "US10Y": "^TNX",
    "US2Y":  "^IRX",
}

# FRED シリーズID（変更不要）
MACRO_FRED = {
    "REAL10Y": "DFII10",
    "SOFR":    "SOFR",
    "IORB":    "IORB",
    "ACMTP10": "ACMTP10",
}

MACRO_LOOKBACK_DAYS = 30

if FRED_API_KEY == "your_fred_api_key_here":
    print("⚠️  FRED APIキー未設定。REAL10Y / SOFR / IORB / ACMTP10 は取得されません。")
    print("   取得先: https://fred.stlouisfed.org/docs/api/api_key.html")
else:
    print(f"✅ FRED APIキー設定済み")
print("✅ マクロ設定完了")

In [ ]:
# Cell 4: テクニカルデータ取得関数

def fetch_ohlcv(symbol: str, timeframe: str, limit: int) -> pd.DataFrame:
    exchange = ccxt.binance({"enableRateLimit": True})
    raw = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
    df = pd.DataFrame(raw, columns=["timestamp", "open", "high", "low", "close", "volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True)
    df.set_index("timestamp", inplace=True)
    return df.astype(float)


def fetch_pair_ohlcv(pair_cfg: dict, timeframe: str, limit: int) -> pd.DataFrame:
    symbols = pair_cfg["binance_symbols"]
    if len(symbols) == 1:
        return fetch_ohlcv(symbols[0], timeframe, limit)

    # レシオペア: BTC/USDT ÷ ETH/USDT = BTC/ETH
    df_num = fetch_ohlcv(symbols[0], timeframe, limit)
    df_den = fetch_ohlcv(symbols[1], timeframe, limit)
    df_num, df_den = df_num.align(df_den, join="inner", axis=0)
    return pd.DataFrame({
        "open":   df_num["open"]  / df_den["open"],
        "high":   df_num["high"]  / df_den["low"],
        "low":    df_num["low"]   / df_den["high"],
        "close":  df_num["close"] / df_den["close"],
        "volume": df_num["volume"],
    }, index=df_num.index)


def fetch_all(pairs: dict, timeframe: str, limit: int) -> dict:
    data = {}
    for key, cfg in pairs.items():
        print(f"  取得中: {cfg['label']}...", end=" ")
        try:
            data[key] = fetch_pair_ohlcv(cfg, timeframe, limit)
            print(f"OK ({len(data[key])} 本)")
        except Exception as e:
            print(f"ERROR: {e}")
            data[key] = None
    return data


print("✅ テクニカルデータ取得関数 定義完了")

In [ ]:
# Cell 5: テクニカル指標計算関数

def calc_indicators(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    bb = ta.bbands(d["close"], length=BB_PERIOD, std=BB_STD)
    d["bb_upper"] = bb[f"BBU_{BB_PERIOD}_{BB_STD}"]
    d["bb_mid"]   = bb[f"BBM_{BB_PERIOD}_{BB_STD}"]
    d["bb_lower"] = bb[f"BBL_{BB_PERIOD}_{BB_STD}"]
    d["bb_width"] = (d["bb_upper"] - d["bb_lower"]) / d["bb_mid"]
    d["bb_pct_b"] = bb[f"BBP_{BB_PERIOD}_{BB_STD}"]

    adx = ta.adx(d["high"], d["low"], d["close"], length=ADX_PERIOD)
    d["adx"]    = adx[f"ADX_{ADX_PERIOD}"]
    d["di_pos"] = adx[f"DMP_{ADX_PERIOD}"]
    d["di_neg"] = adx[f"DMN_{ADX_PERIOD}"]

    atr = ta.atr(d["high"], d["low"], d["close"], length=ATR_PERIOD)
    d["atr"]     = atr
    d["atr_pct"] = (atr / d["close"]) * 100.0

    return d.dropna()


def calc_range_score(df: pd.DataFrame) -> dict:
    last    = df.iloc[-1]
    adx_now = last["adx"]
    bbw_now = last["bb_width"]
    bbw_hist = df["bb_width"]

    adx_score = 2 if adx_now < 15 else (1 if adx_now < 25 else 0)

    # BB幅を過去のパーセンタイルと比較（絶対値ではなく相対評価）
    bbw_score = (2 if bbw_now <= bbw_hist.quantile(0.33)
                 else (1 if bbw_now <= bbw_hist.quantile(0.66) else 0))

    score_5 = adx_score + bbw_score + 1  # 1〜5
    labels  = {1: "STRONG TREND", 2: "TREND", 3: "NEUTRAL", 4: "RANGING", 5: "STRONG RANGE"}

    return {
        "score":    score_5,
        "label":    labels[score_5],
        "adx":      round(adx_now, 1),
        "bb_width": round(bbw_now * 100, 2),
        "atr_pct":  round(last["atr_pct"], 2),
    }


def calc_lp_consumption(df: pd.DataFrame, lp_lower: float, lp_upper: float) -> float:
    price     = df["close"].iloc[-1]
    center    = (lp_upper + lp_lower) / 2.0
    half_range = (lp_upper - lp_lower) / 2.0
    if half_range == 0:
        return 0.0
    return abs(price - center) / half_range * 100.0


print("✅ テクニカル指標計算関数 定義完了")

In [ ]:
# Cell 6: マクロデータ取得・スコア計算関数

def fetch_macro_yfinance(symbols: dict, lookback_days: int) -> dict:
    end   = datetime.now(timezone.utc)
    start = end - timedelta(days=lookback_days + 5)
    data  = {}
    for name, ticker in symbols.items():
        try:
            df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
            if df.empty:
                raise ValueError("empty response")
            if df.index.tz is None:
                df.index = df.index.tz_localize("UTC")
            close = df["Close"]
            if isinstance(close, pd.DataFrame):
                close = close.iloc[:, 0]
            data[name] = close.rename(name)
        except Exception as e:
            print(f"  yfinance {name} ({ticker}): ERROR {e}")
            data[name] = None
    return data


def fetch_macro_fred(series: dict, api_key: str, lookback_days: int) -> dict:
    if api_key == "your_fred_api_key_here":
        return {k: None for k in series}
    fred  = Fred(api_key=api_key)
    start = datetime.now() - timedelta(days=lookback_days + 10)
    data  = {}
    for name, series_id in series.items():
        try:
            s = fred.get_series(series_id, observation_start=start)
            s.index = pd.to_datetime(s.index).tz_localize("UTC")
            data[name] = s.rename(name)
        except Exception as e:
            print(f"  FRED {name} ({series_id}): ERROR {e}")
            data[name] = None
    return data


def calc_macro_scores(yf_data: dict, fred_data: dict, cme_hike_prob_ye: float) -> dict:
    scores = {}

    vix = yf_data.get("VIX")
    if vix is not None and len(vix) > 0:
        v = float(vix.iloc[-1])
        scores["VIX"] = {"val": round(v, 2), "unit": "",
                         "level": "HIGH" if v > 25 else ("MED" if v > 16 else "LOW")}

    dxy = yf_data.get("DXY")
    if dxy is not None and len(dxy) >= 5:
        chg = (float(dxy.iloc[-1]) - float(dxy.iloc[-5])) / float(dxy.iloc[-5]) * 100
        scores["DXY_5d"] = {"val": round(chg, 2), "unit": "%",
                             "level": "HIGH" if abs(chg) > 1.5 else ("MED" if abs(chg) > 0.5 else "LOW")}

    real10y = fred_data.get("REAL10Y")
    if real10y is not None and len(real10y) > 0:
        r = float(real10y.dropna().iloc[-1])
        scores["REAL10Y"] = {"val": round(r, 3), "unit": "%",
                              "level": "HIGH" if r > 2.5 else ("MED" if r > 2.0 else "LOW")}

    sofr = fred_data.get("SOFR")
    iorb = fred_data.get("IORB")
    if sofr is not None and iorb is not None:
        combined = pd.concat([sofr.dropna(), iorb.dropna()], axis=1).dropna()
        if len(combined) > 0:
            spread_bp = (float(combined.iloc[-1, 0]) - float(combined.iloc[-1, 1])) * 100
            scores["SOFR_IORB"] = {"val": round(spread_bp, 1), "unit": "bp",
                                    "level": "HIGH" if spread_bp > 3 else ("MED" if spread_bp > 0 else "LOW")}

    tp = fred_data.get("ACMTP10")
    if tp is not None and len(tp) > 0:
        t = float(tp.dropna().iloc[-1])
        scores["ACMTP10"] = {"val": round(t, 4), "unit": "%",
                              "level": "HIGH" if t > 0.8 else ("MED" if t > 0.5 else "LOW")}

    scores["CME_YE"] = {"val": cme_hike_prob_ye, "unit": "%",
                         "level": "HIGH" if cme_hike_prob_ye > 60 else ("MED" if cme_hike_prob_ye > 30 else "LOW")}

    today = datetime.now().date()
    fomc_dates_parsed = [datetime.strptime(d, "%Y-%m-%d").date() for d in FOMC_DATES]
    days_to_next = min((abs((d - today).days) for d in fomc_dates_parsed), default=999)
    scores["FOMC_DAYS"] = {"val": days_to_next, "unit": "日",
                            "level": "HIGH" if days_to_next <= 3 else ("MED" if days_to_next <= 7 else "LOW")}

    high_count = sum(1 for v in scores.values() if v["level"] == "HIGH")
    scores["__OVERALL__"] = "HIGH" if high_count >= 3 else ("MED" if high_count >= 1 else "LOW")

    return scores


print("✅ マクロデータ取得・スコア計算関数 定義完了")

In [ ]:
# Cell 7: チャート構築関数

SCORE_COLORS = {
    "STRONG RANGE": "#00C853",
    "RANGING":      "#69F0AE",
    "NEUTRAL":      "#FFD600",
    "TREND":        "#FF6D00",
    "STRONG TREND": "#D50000",
}
LEVEL_COLORS = {"LOW": "#00C853", "MED": "#FFD600", "HIGH": "#D50000"}


def make_pair_chart(df: pd.DataFrame, pair_cfg: dict, score_info: dict) -> go.Figure:
    lp_lower = pair_cfg["lp_lower"]
    lp_upper = pair_cfg["lp_upper"]
    lp_mid   = (lp_lower + lp_upper) / 2.0
    lp_pct   = calc_lp_consumption(df, lp_lower, lp_upper)

    score_color  = SCORE_COLORS.get(score_info["label"], "#FFFFFF")
    lp_pct_warn  = " ⚠️ レンジアウト" if lp_pct > 100 else (" ⚠️ エッジ接近" if lp_pct > 80 else "")

    title_text = (
        f"{pair_cfg['label']}  |  "
        f"スコア {score_info['score']}/5 [{score_info['label']}]  |  "
        f"ADX: {score_info['adx']}  BB幅: {score_info['bb_width']}%  ATR: {score_info['atr_pct']}%  |  "
        f"LP消費: {lp_pct:.1f}%{lp_pct_warn}"
    )

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        row_heights=[0.55, 0.25, 0.20],
        subplot_titles=["価格 + BB + LP レンジ", "ADX / DI", "ATR %"],
    )

    # ── Row 1: ローソク足
    fig.add_trace(go.Candlestick(
        x=df.index,
        open=df["open"], high=df["high"], low=df["low"], close=df["close"],
        name="OHLC",
        increasing_line_color="#26A69A", decreasing_line_color="#EF5350",
        showlegend=False,
    ), row=1, col=1)

    # Bollinger Band（塗り）
    fig.add_trace(go.Scatter(
        x=df.index, y=df["bb_upper"],
        line=dict(color="rgba(100,149,237,0.6)", width=1),
        name="BB Upper", showlegend=False,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=df.index, y=df["bb_lower"],
        line=dict(color="rgba(100,149,237,0.6)", width=1),
        fill="tonexty", fillcolor="rgba(100,149,237,0.08)",
        name="BB Band", showlegend=False,
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=df.index, y=df["bb_mid"],
        line=dict(color="rgba(100,149,237,0.4)", width=1, dash="dot"),
        name="BB Mid", showlegend=False,
    ), row=1, col=1)

    # LP レンジ帯
    x_range = [df.index[0], df.index[-1]]
    fig.add_trace(go.Scatter(
        x=x_range, y=[lp_upper, lp_upper],
        line=dict(color="rgba(0,200,83,0.9)", width=1.5, dash="dash"),
        name="LP Upper",
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=x_range, y=[lp_lower, lp_lower],
        line=dict(color="rgba(0,200,83,0.9)", width=1.5, dash="dash"),
        fill="tonexty", fillcolor="rgba(0,200,83,0.10)",
        name="LP Range",
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=x_range, y=[lp_mid, lp_mid],
        line=dict(color="rgba(0,200,83,0.4)", width=1, dash="dot"),
        name="LP Center", showlegend=False,
    ), row=1, col=1)

    # FOMC イベント縦線（価格チャートのみ）
    for date_str in FOMC_DATES:
        fomc_dt = pd.Timestamp(date_str, tz="UTC")
        if df.index[0] <= fomc_dt <= df.index[-1]:
            fig.add_vline(
                x=fomc_dt.timestamp() * 1000,
                line_dash="dash",
                line_color="rgba(255,80,80,0.7)",
                line_width=1.5,
                row=1, col=1,
            )

    # ── Row 2: ADX
    fig.add_trace(go.Scatter(
        x=df.index, y=df["adx"],
        line=dict(color="#F57F17", width=2), name="ADX",
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=df.index, y=df["di_pos"],
        line=dict(color="#43A047", width=1, dash="dot"), name="DI+",
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=df.index, y=df["di_neg"],
        line=dict(color="#E53935", width=1, dash="dot"), name="DI-",
    ), row=2, col=1)
    fig.add_hline(y=25, line_dash="dash", line_color="gray", line_width=1, row=2, col=1)
    fig.add_annotation(
        x=df.index[-1], y=25, text="25",
        showarrow=False, xanchor="right",
        font=dict(size=9, color="gray"),
        row=2, col=1,
    )

    # ── Row 3: ATR%
    fig.add_trace(go.Scatter(
        x=df.index, y=df["atr_pct"],
        line=dict(color="#7B1FA2", width=1.5), name="ATR %",
        fill="tozeroy", fillcolor="rgba(123,31,162,0.12)",
    ), row=3, col=1)

    fig.update_layout(
        title=dict(text=title_text, font=dict(size=13, color=score_color)),
        height=700,
        margin=dict(l=60, r=20, t=70, b=30),
        xaxis_rangeslider_visible=False,
        template="plotly_dark",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )
    fig.update_yaxes(title_text="Price", row=1, col=1)
    fig.update_yaxes(title_text="ADX / DI", row=2, col=1)
    fig.update_yaxes(title_text="ATR %", row=3, col=1)

    return fig


def build_all_charts(all_data: dict, pairs: dict) -> list:
    results = []
    for key, df in all_data.items():
        if df is None:
            continue
        df_ind = calc_indicators(df)
        score  = calc_range_score(df_ind)
        score["lp_pct"] = calc_lp_consumption(df_ind, pairs[key]["lp_lower"], pairs[key]["lp_upper"])
        results.append((key, make_pair_chart(df_ind, pairs[key], score), score))
    return results


def make_macro_panel(
    yf_data: dict,
    fred_data: dict,
    macro_scores: dict,
    fomc_dates: list,
    lookback_days: int,
) -> go.Figure:
    overall  = macro_scores.get("__OVERALL__", "N/A")
    bg_color = {"LOW": "rgba(0,200,83,0.12)", "MED": "rgba(255,214,0,0.12)",
                "HIGH": "rgba(213,0,0,0.15)"}.get(overall, "rgba(50,50,50,0.3)")
    title_color = LEVEL_COLORS.get(overall, "#FFFFFF")

    fig = make_subplots(
        rows=2, cols=2,
        shared_xaxes=False,
        subplot_titles=["DXY（ドル指数）", "VIX（株式ボラ）", "WTI 原油", "Gold 金"],
        vertical_spacing=0.12,
        horizontal_spacing=0.08,
    )

    plot_map = [
        ("DXY",  1, 1, "#64B5F6"),
        ("VIX",  1, 2, "#EF5350"),
        ("WTI",  2, 1, "#FF8F00"),
        ("Gold", 2, 2, "#FFD600"),
    ]

    for name, row, col, color in plot_map:
        s = yf_data.get(name)
        if s is not None and len(s) > 0:
            s_tail = s.tail(lookback_days)
            fig.add_trace(go.Scatter(
                x=s_tail.index, y=s_tail.values,
                line=dict(color=color, width=1.8),
                name=name, showlegend=False,
                fill="tozeroy",
                fillcolor=color.replace("#", "rgba(").rstrip(")") + ",0.08)" if "#" in color else color,
            ), row=row, col=col)

    # VIX 25 の警戒ライン
    vix_s = yf_data.get("VIX")
    if vix_s is not None:
        fig.add_hline(y=25, line_dash="dash", line_color="rgba(255,80,80,0.6)",
                      line_width=1, row=1, col=2)

    # スコアテーブルをアノテーションで表示
    score_lines = []
    label_map = {
        "VIX":        "VIX",
        "DXY_5d":     "DXY 5日変化",
        "REAL10Y":    "10Y 実質金利",
        "SOFR_IORB":  "SOFR-IORB",
        "ACMTP10":    "Term Premia",
        "CME_YE":     "CME年末利上げ",
        "FOMC_DAYS":  "FOMC まで",
    }
    for key, label in label_map.items():
        if key in macro_scores:
            info  = macro_scores[key]
            lv    = info["level"]
            emoji = {"LOW": "🟢", "MED": "🟡", "HIGH": "🔴"}.get(lv, "⚪")
            score_lines.append(f"{emoji} {label}: {info['val']}{info['unit']}")
        else:
            score_lines.append(f"⚪ {label_map[key]}: N/A (FRED未設定)")

    score_text = "<br>".join(score_lines)

    fig.update_layout(
        title=dict(
            text=f"🌐 マクロ環境パネル  |  総合リスク: <b>{overall}</b>  ({datetime.now().strftime('%Y/%m/%d %H:%M')} 更新)",
            font=dict(size=15, color=title_color),
        ),
        height=500,
        template="plotly_dark",
        paper_bgcolor=bg_color,
        margin=dict(l=50, r=280, t=70, b=30),
        annotations=[
            dict(
                x=1.02, y=0.98,
                xref="paper", yref="paper",
                text=score_text,
                showarrow=False,
                align="left",
                font=dict(size=12, family="monospace"),
                bgcolor="rgba(30,30,30,0.8)",
                bordercolor="rgba(200,200,200,0.3)",
                borderwidth=1,
                xanchor="left",
                yanchor="top",
            )
        ],
    )

    return fig


print("✅ チャート構築関数 定義完了")

In [ ]:
# Cell 8: マクロデータ取得実行

print("マクロデータ取得中...")
print("  [yfinance]")
yf_data = fetch_macro_yfinance(MACRO_YFINANCE, MACRO_LOOKBACK_DAYS)
for name, s in yf_data.items():
    if s is not None:
        print(f"    {name}: OK ({len(s)} 件, 最新: {float(s.iloc[-1]):.4f})")

print("  [FRED]")
fred_data = fetch_macro_fred(MACRO_FRED, FRED_API_KEY, MACRO_LOOKBACK_DAYS)
for name, s in fred_data.items():
    if s is not None:
        print(f"    {name}: OK ({len(s)} 件, 最新: {float(s.dropna().iloc[-1]):.4f})")
    else:
        print(f"    {name}: N/A")

macro_scores = calc_macro_scores(yf_data, fred_data, CME_RATE_HIKE_PROB_YE)

print("\n── マクロスコアサマリー ──")
for k, v in macro_scores.items():
    if k != "__OVERALL__":
        emoji = {"LOW": "🟢", "MED": "🟡", "HIGH": "🔴"}.get(v["level"], "⚪")
        print(f"  {emoji} {k}: {v['val']}{v['unit']}  [{v['level']}]")
overall = macro_scores["__OVERALL__"]
print(f"\n  総合マクロリスク: {overall}")
print("\n✅ マクロデータ取得完了")

In [ ]:
# Cell 9: 全チャート表示（メイン実行）

# ── テクニカルデータ取得
print(f"テクニカルデータ取得中 ({TIMEFRAME}, {CANDLE_LIMIT}本)...")
all_data = fetch_all(PAIRS, TIMEFRAME, CANDLE_LIMIT)

# ── サマリーテーブル
charts = build_all_charts(all_data, PAIRS)

print("\n" + "=" * 72)
print(f"{'ペア':<20} {'スコア':>6} {'判定':<14} {'ADX':>6} {'BB幅%':>7} {'ATR%':>6} {'LP消費%':>8}")
print("-" * 72)
for key, fig, score in charts:
    lp_warn = " ⚠️" if score["lp_pct"] > 80 else ""
    print(
        f"{PAIRS[key]['label']:<20} "
        f"{score['score']:>6} "
        f"{score['label']:<14} "
        f"{score['adx']:>6.1f} "
        f"{score['bb_width']:>7.2f} "
        f"{score['atr_pct']:>6.2f} "
        f"{score['lp_pct']:>7.1f}%{lp_warn}"
    )
print("=" * 72)
print("LP消費%: レンジ中心からの距離。80%超=エッジ接近、100%超=レンジアウト")
print(f"マクロ総合リスク: {macro_scores['__OVERALL__']}\n")

# ── マクロパネル表示
macro_fig = make_macro_panel(yf_data, fred_data, macro_scores, FOMC_DATES, MACRO_LOOKBACK_DAYS)
macro_fig.show()

# ── テクニカルチャート × 4ペア表示
for key, fig, score in charts:
    fig.show()